## This notebook deals with PySpark code detailing how to deal with different data types such as Numbers, Booleans, Strings, Dates and Timestamps & Null values (how to identify them and fill them out)

In [0]:
df=spark.read.table("samples.bakehouse.sales_transactions")
df.display()

In [0]:
#outer quotes behave like sql so paymentMethod=='mastercard' becomes a sql expression and inner quotes define the mastercard as a string literal inside the sql expression
# df2=df.where("paymentMethod=='mastercard'")
#another way of writing the same would be
from pyspark.sql.functions import col
# df2=df.where(col("paymentMethod")=="mastercard")
df2=df.filter(col("paymentMethod")=="mastercard")
# df2=df.where("paymentMethod"!="mastercard")
df2.display()


In [0]:
%sql
select * from samples.bakehouse.sales_transactions where paymentMethod='mastercard'
-- select * from samples.bakehouse.sales_transactions where paymentMethod!='mastercard'

In [0]:
from pyspark.sql.functions import expr, pow, round
# df2=df.withColumn("myCol",pow("quantity",2)/6)
df2=df.withColumn("myCol",round(pow("quantity",2)/6,2))
#writing the same using expr keyword
# df2 = df.withColumn("myCol", expr("round(pow(quantity, 2) / 6, 2)"))
display(df2)

In [0]:
%sql
select *,round(pow(quantity,2)/6,2) from samples.bakehouse.sales_transactions

In [0]:
from pyspark.sql.functions import lower, expr
df2=df.select("product").\
withColumn("lower_product",lower("product")).\
withColumn("upper_product",col("product")).\
withColumn("trim_product",expr("trim(product)")).\
withColumn("substr_product",expr("substr(product,0,4)")).\
withColumn("split_product",expr("split(product,' ')"))
display(df2)

In [0]:
%sql
select 
product, lower(product) as lower_product,
upper(product) as upper_product,
trim(product) as trim_product,
substr(product,0,4) as substr_product,
split(product,' ') as split_product
from samples.bakehouse.sales_transactions

If the pattern doesn't match, **regexp_extract()** returns an empty string.
1. regexp_extract() → extracts the first match.

2. regexp_extract_all() → extracts all matches into an array.

3. regexp_replace() → replaces all occurrences of the matched pattern with the replacement string.

In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace
df2=df.select("product").\
withColumn("regexp_extract_product", expr("regexp_extract(product, r'^\S+\s+(\S+)\s+\S+$', 1)")).\
withColumn("regexp_replace_product", expr("regexp_replace(product, r'^\S+\s+(\S+)\s+\S+$', 'AAAA')"))
display(df2)

In [0]:
%sql
SELECT 
product,
regexp_extract(product, r'^\S+\s+(\S+)\s+\S+$', 1) AS regexp_extract_product,
regexp_replace(product, r'^\S+\s+(\S+)\s+\S+$', 'AAAA') AS regexp_replace_product
from samples.bakehouse.sales_transactions

In [0]:
# from pyspark.sql.functions import current_date, current_timestamp, to_date,date_add,date_sub,lit,to_timestamp

df2 = df.select("dateTime")\
.withColumn("today", current_date())\
.withColumn("now", current_timestamp())\
.withColumn("dateTime_date", to_date("dateTime"))\
.withColumn("dateTime_date_add", date_add(to_date("dateTime"),5))\
.withColumn("dateTime_date_sub", date_sub(to_date("dateTime"),5))\
.withColumn("custom_date", to_date(lit("20251014"),'yyyyMMdd'))\
.withColumn("custom_timestamp", to_timestamp(lit("20251014112233"),'yyyyMMddHHmmss'))
display(df2)

In [0]:
%sql
select 
    dateTime,
    current_date() as today,
    current_timestamp() as now,
    to_date(dateTime) as dateTime_date,
    date_add(dateTime, 1) as dateTime_date_add,
    date_sub(dateTime, 1) as dateTime_date_sub,
    to_date(dateTime, 'yyyyMMdd') as custom_date,
    to_timestamp(dateTime, 'yyyyMMddHHmmss') as custom_timestamp
from samples.bakehouse.sales_transactions

1. coalesce here detects the first non-null column and uses it for displaying. So basically out of transactionID and customerID whichever will be the first non-null column value we display it, if both are null it returns null
2. same thing we are doing use case statements conditonal logic
3. .na.drop(subset=["column_name"]) basically drops the row where the specific column_name we specify has a null value
4. .na.fill('myvalue') fills the empty null value row in all columns with this myvalue.
5. .na.fill({'transactionID': 0, 'customerID': 0}) fills the null values of the 2 id columns with custom vals
6. .na.fill("myvalue",subset["customerID"]) this fills null values of customerID column with myvalue


In [0]:
from pyspark.sql.functions import coalesce, expr
df2 = df.select("transactionID","customerID")\
.withColumn("coalesce_id", coalesce("transactionID", "customerID"))\
.withColumn("if_else", expr("CASE WHEN transactionID IS NOT NULL THEN transactionID ELSE customerID END"))\
.na.drop(subset=["transactionID"])
# .na.drop('all')
# .na.fill('myvalue')
# .na.fill({'transactionID': 0, 'customerID': 0})
# .na.fill("myvalue",subset=["customerID"])


In [0]:
%sql
SELECT
    transactionID,
    customerID,
    coalesce(transactionID, customerID) AS coalesce_id,
    CASE
        WHEN transactionID IS NOT NULL THEN transactionID
        ELSE customerID
    END AS if_else
FROM samples.bakehouse.sales_transactions;